In [ ]:
# ==========================================================
# 1. Imports
# ==========================================================

import pandas as pd
import numpy as np

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

In [ ]:
# ==========================================================
# 2. Load Cleaned Dataset
# ==========================================================

df = pd.read_csv(
    r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_london_all_years_decoded.csv",
    low_memory=False
)

print(f"Dataset Shape: {df.shape}")

display(df.head())

In [ ]:
# ==========================================================
# 3. Create Activity Level
# ==========================================================

df["Activity_Level"] = pd.cut(
    df["MEMS7_ALL"],
    bins=[-1, 29, 149, float("inf")],
    labels=["Inactive", "Fairly Active", "Active"]
)

print("Activity_Level created successfully.")

display(
    df[["MEMS7_ALL", "Activity_Level"]].head(10)
)

In [ ]:
df["Activity_Level"].value_counts(dropna=False)

In [ ]:
df["Activity_Level"] = pd.Categorical(
    df["Activity_Level"],
    categories=["Inactive", "Fairly Active", "Active"],
    ordered=True
)

In [ ]:
df["Age9"].value_counts().sort_index()

In [ ]:
# ==========================================================
# 4. Create Age_Group
# ==========================================================

age_mapping = {
    "16-24": "16-24",
    "25-34": "25-44",
    "35-44": "25-44",
    "45-54": "45-64",
    "55-64": "45-64",
    "65-74": "65+",
    "75-84": "65+",
    "85+": "65+"
}

df["Age_Group"] = df["Age9"].map(age_mapping)

print("Age_Group created successfully.")

display(
    df[["Age9", "Age_Group"]].head(10)
)

In [ ]:
df["Age_Group"].value_counts().sort_index()

In [ ]:
# ==========================================================
# 5. Select Final Modelling Variables
# ==========================================================

model_df = df[
    [
        # Demographics
        "survey_year",
        "borough",
        "Age_Group",
        "Gend3",
        "Eth7",
        "Disab3",
        "Educ6",
        "health",
        "wt_final",

        # Activity Level Model
        "MEMS7_ALL",
        "Activity_Level",

        # Activity Type Model
        "MONTHS_12_WALKALL_C01",
        "MONTHS_12_CYCALL_C02",
        "MONTHS_12_ACTTRAV_C03",
        "MONTHS_12_DANCEALL_C04",
        "MONTHS_12_TEAMSPORT_C05",
        "MONTHS_12_RACKETSPORT_C06",
        "MONTHS_12_ADVWATERSPORT_C07",
        "MONTHS_12_LEISURE_C08",
        "MONTHS_12_COMBATTARGET_C09"
    ]
].copy()

print(f"Final modelling dataset shape: {model_df.shape}")

display(model_df.head())

In [ ]:
# ==========================================================
# 6. Missing Values in Final Dataset
# ==========================================================

missing = (
    model_df.isna()
            .sum()
            .to_frame("Missing")
)

missing["Percent"] = (
    missing["Missing"] / len(model_df) * 100
).round(2)

display(
    missing[missing["Missing"] > 0]
    .sort_values("Percent", ascending=False)
)

In [ ]:
cols = [
    "health",
    "Eth7",
    "Disab3",
    "Educ6",
    "MONTHS_12_CYCALL_C02"
]

for col in cols:
    print(f"\n{col}")
    display(pd.crosstab(model_df["survey_year"],
                        model_df[col].isna(),
                        normalize="index") * 100)

In [ ]:
activity_cols = [
    "MONTHS_12_WALKALL_C01",
    "MONTHS_12_CYCALL_C02",
    "MONTHS_12_ACTTRAV_C03",
    "MONTHS_12_DANCEALL_C04",
    "MONTHS_12_TEAMSPORT_C05",
    "MONTHS_12_RACKETSPORT_C06",
    "MONTHS_12_ADVWATERSPORT_C07",
    "MONTHS_12_LEISURE_C08",
    "MONTHS_12_COMBATTARGET_C09"
]

for col in activity_cols:
    print(f"\n{col}")
    print(model_df[col].isna().mean() * 100)

In [ ]:
pd.crosstab(
    df["survey_year"],
    df["MONTHS_12_CYCALL_C02"].isna(),
    margins=True
)

In [ ]:
model_df.dtypes

In [ ]:
# ==========================================================
# 7. Convert Variables to Categorical
# ==========================================================

categorical_cols = [
    "survey_year",
    "borough",
    "Age_Group",
    "Gend3",
    "Eth7",
    "Disab3",
    "Educ6",
    "health",
    "Activity_Level",

    "MONTHS_12_WALKALL_C01",
    "MONTHS_12_CYCALL_C02",
    "MONTHS_12_ACTTRAV_C03",
    "MONTHS_12_DANCEALL_C04",
    "MONTHS_12_TEAMSPORT_C05",
    "MONTHS_12_RACKETSPORT_C06",
    "MONTHS_12_ADVWATERSPORT_C07",
    "MONTHS_12_LEISURE_C08",
    "MONTHS_12_COMBATTARGET_C09"
]

for col in categorical_cols:
    model_df[col] = model_df[col].astype("category")

print("Categorical conversion completed.")

In [ ]:
model_df.dtypes

In [ ]:
# ==========================================================
# 8. Save Prepared Modelling Dataset
# ==========================================================

output_path = r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data\active_lives_modelling_dataset.csv"

model_df.to_csv(output_path, index=False)

print(f"Dataset saved successfully.\n")
print(f"Location:\n{output_path}")
print(f"\nFinal Shape: {model_df.shape}")

In [ ]:
# ==========================================================
# Save Prepared Modelling Dataset (Parquet)
# ==========================================================

parquet_path = (
    r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data"
    r"\active_lives_modelling_dataset.parquet"
)

model_df.to_parquet(
    parquet_path,
    engine="pyarrow",
    index=False
)

print("Parquet dataset saved successfully.")
print(f"Location:\n{parquet_path}")

In [ ]:
# ==========================================================
# Verify Parquet File
# ==========================================================

model_df_parquet = pd.read_parquet(
    parquet_path,
    engine="pyarrow"
)

print(f"Shape: {model_df_parquet.shape}")

display(model_df_parquet.head())

In [ ]:
import os

csv_path = (
    r"C:\Users\trish\OneDrive - University of Bristol\summer project\TS\Data"
    r"\active_lives_modelling_dataset.csv"
)

print(f"CSV Size     : {os.path.getsize(csv_path) / (1024**2):.2f} MB")
print(f"Parquet Size : {os.path.getsize(parquet_path) / (1024**2):.2f} MB")

In [ ]:
# ==========================================================
# Verify Data Types After Loading Parquet
# ==========================================================

model_df_parquet.dtypes

In [ ]:
# ==========================================================
# Activity Level Trend by Survey Year
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt

# Calculate percentage distribution by year
activity_trend = (
    pd.crosstab(
        model_df["survey_year"],
        model_df["Activity_Level"],
        normalize="index"
    ) * 100
)

# Ensure consistent order
activity_trend = activity_trend[
    ["Inactive", "Fairly Active", "Active"]
]

display(activity_trend.round(2))

# Plot
plt.figure(figsize=(10,6))

for level in activity_trend.columns:
    plt.plot(
        activity_trend.index,
        activity_trend[level],
        marker="o",
        linewidth=2,
        label=level
    )

plt.title("Activity Level Trend by Survey Year", fontsize=14)
plt.xlabel("Survey Year")
plt.ylabel("Percentage of Respondents")
plt.ylim(0,100)
plt.grid(alpha=0.3)
plt.legend(title="Activity Level")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()